# Data Preprocessing
This notebook handles all preprocessing steps including missing value treatment, feature encoding, and data preparation for modeling.
All steps are applied after the train-validation-test split to prevent data leakage.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the saved raw data
file_path = "C:\Courses\DAMG6105\Project - Disease Prediction System"
df = pd.read_csv(f'{file_path}/data/processed/heart_disease_raw.csv')

print("Data loaded successfully.")
print("Shape:", df.shape)
df.head()

Data loaded successfully.
Shape: (920, 16)


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [3]:
# Inspect the two high-missingness columns
print("=== ca column ===")
print("Data type:", df['ca'].dtype)
print(df['ca'].value_counts(dropna=False))

print("\n=== thal column ===")
print("Data type:", df['thal'].dtype)
print(df['thal'].value_counts(dropna=False))

=== ca column ===
Data type: float64
ca
NaN    611
0.0    181
1.0     67
2.0     41
3.0     20
Name: count, dtype: int64

=== thal column ===
Data type: object
thal
NaN                  486
normal               196
reversable defect    192
fixed defect          46
Name: count, dtype: int64


In [4]:
# Add missing indicators for high-missingness columns ca & thal
df['ca_missing'] = df['ca'].isnull().astype(int) 
df['thal_missing'] = df['thal'].isnull().astype(int)

# Verify
print("ca_missing distribution:")
print(df['ca_missing'].value_counts())

print("\nthal_missing distribution:")
print(df['thal_missing'].value_counts())

print("\nNew shape:", df.shape)

ca_missing distribution:
ca_missing
1    611
0    309
Name: count, dtype: int64

thal_missing distribution:
thal_missing
1    486
0    434
Name: count, dtype: int64

New shape: (920, 18)


In [21]:
# Encode thal: convert text categories to numbers
thal_mapping = {
    'normal': 0,
    'fixed defect': 1,
    'reversable defect': 2
}

df['thal'] = df['thal'].map(thal_mapping)

# Verify
print(df['thal'].value_counts(dropna=False))

thal
NaN    486
0.0    196
2.0    192
1.0     46
Name: count, dtype: int64


In [22]:
print(df.dtypes)
print(df.select_dtypes(include='object').columns.tolist())


id                int64
age               int64
sex              object
dataset          object
cp               object
trestbps        float64
chol            float64
fbs              object
restecg          object
thalch          float64
exang            object
oldpeak         float64
slope            object
ca              float64
thal            float64
num               int64
ca_missing        int64
thal_missing      int64
dtype: object
['sex', 'dataset', 'cp', 'fbs', 'restecg', 'exang', 'slope']


In [23]:
print(df['dataset'].value_counts())

dataset
Cleveland        304
Hungary          293
VA Long Beach    200
Switzerland      123
Name: count, dtype: int64


In [24]:
""" 
Drop dataset column - it is a data source identifier, not a clinical feature.
Including it would teach the model "patients from Cleveland tend to have heart disease" rather than learning actual medical patterns,
which is both misleading and useless for a general prediction system.
"""
df = df.drop(columns=['dataset'])
print("'dataset' column dropped.")
print("New shape:", df.shape)


'dataset' column dropped.
New shape: (920, 17)


In [25]:
print("\nRemaining object columns:")
print(df.select_dtypes(include='object').columns.tolist())


Remaining object columns:
['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope']


In [26]:
# Drop id column - row identifier, not a clinical feature
df = df.drop(columns=['id'])

print("'id' column dropped.")
print("New shape:", df.shape)

'id' column dropped.
New shape: (920, 16)


In [27]:
# Check unique values for all remaining object columns
obj_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope']

for col in obj_cols:
    print(f"=== {col} ===")
    print(df[col].value_counts(dropna=False))
    print()

=== sex ===
sex
Male      726
Female    194
Name: count, dtype: int64

=== cp ===
cp
asymptomatic       496
non-anginal        204
atypical angina    174
typical angina      46
Name: count, dtype: int64

=== fbs ===
fbs
False    692
True     138
NaN       90
Name: count, dtype: int64

=== restecg ===
restecg
normal              551
lv hypertrophy      188
st-t abnormality    179
NaN                   2
Name: count, dtype: int64

=== exang ===
exang
False    528
True     337
NaN       55
Name: count, dtype: int64

=== slope ===
slope
flat           345
NaN            309
upsloping      203
downsloping     63
Name: count, dtype: int64



In [28]:
# Encode binary columns
df['sex'] = df['sex'].map({'Male': 1, 'Female': 0})
df['fbs'] = df['fbs'].map({True: 1, False: 0})
df['exang'] = df['exang'].map({True: 1, False: 0})

# Verify
print("sex:", df['sex'].value_counts(dropna=False).to_dict())
print("fbs:", df['fbs'].value_counts(dropna=False).to_dict())
print("exang:", df['exang'].value_counts(dropna=False).to_dict())

sex: {1: 726, 0: 194}
fbs: {0.0: 692, 1.0: 138, nan: 90}
exang: {0.0: 528, 1.0: 337, nan: 55}


In [29]:
# Encode multi-category columns
df['cp'] = df['cp'].map({
    'typical angina': 0,
    'atypical angina': 1,
    'non-anginal': 2,
    'asymptomatic': 3
})

df['restecg'] = df['restecg'].map({
    'normal': 0,
    'st-t abnormality': 1,
    'lv hypertrophy': 2
})

df['slope'] = df['slope'].map({
    'upsloping': 0,
    'flat': 1,
    'downsloping': 2
})

# Verify
print("cp:", df['cp'].value_counts(dropna=False).to_dict())
print("restecg:", df['restecg'].value_counts(dropna=False).to_dict())
print("slope:", df['slope'].value_counts(dropna=False).to_dict())

cp: {3: 496, 2: 204, 1: 174, 0: 46}
restecg: {0.0: 551, 2.0: 188, 1.0: 179, nan: 2}
slope: {1.0: 345, nan: 309, 0.0: 203, 2.0: 63}


In [30]:
from sklearn.model_selection import train_test_split

X = df.drop('num', axis=1)
y = df['num']

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Split the 30% into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Split complete.")
print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)

Split complete.
Train size: (644, 15)
Validation size: (138, 15)
Test size: (138, 15)


In [31]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)

# Fit only on training data
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns
)

# Apply to validation and test - do not fit again
X_val_imputed = pd.DataFrame(
    imputer.transform(X_val),
    columns=X_val.columns
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

print("Imputation complete.")
print("Missing values remaining in X_train:", X_train_imputed.isnull().sum().sum())
print("Missing values remaining in X_val:", X_val_imputed.isnull().sum().sum())
print("Missing values remaining in X_test:", X_test_imputed.isnull().sum().sum())

Imputation complete.
Missing values remaining in X_train: 0
Missing values remaining in X_val: 0
Missing values remaining in X_test: 0


## Cholesterol Outlier Treatment (Winsorization)

### Discovery
During Exploratory Data Analysis (Notebook 03), cholesterol was found to have approximately 20% outliers (~129 out of 644 training rows) - significantly higher than any other feature. Extreme values (500+) were identified as likely measurement errors across the multiple hospital sources in the combined dataset. This also explained the counterintuitive negative correlation between cholesterol and disease
risk observed during EDA.

### Decision
Rather than removing 20% of training rows, Winsorization (capping) was applied - preserving all rows while limiting the influence of extreme values. Bounds were calculated exclusively from training data using the IQR method to prevent data leakage, then applied to validation and test sets using the same training bounds.

### Bounds
- Lower bound: 30.9
- Upper bound: 407.1

In [32]:
import pickle

# Calculate and save Winsorization bounds BEFORE scaling
Q1 = X_train_imputed['chol'].quantile(0.25)
Q3 = X_train_imputed['chol'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

chol_bounds = {'lower': lower_bound, 'upper': upper_bound}

print(f"Cholesterol bounds calculated:")
print(f"Lower: {lower_bound:.1f}")
print(f"Upper: {upper_bound:.1f}")

Cholesterol bounds calculated:
Lower: 30.9
Upper: 407.1


In [34]:
from sklearn.preprocessing import StandardScaler

# Define columns to scale - continuous numerical features only
cols_to_scale = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']

# Initialize scaler
scaler = StandardScaler()

# Fit ONLY on training data, then apply to all three sets
X_train_imputed[cols_to_scale] = scaler.fit_transform(X_train_imputed[cols_to_scale])
X_val_imputed[cols_to_scale] = scaler.transform(X_val_imputed[cols_to_scale])
X_test_imputed[cols_to_scale] = scaler.transform(X_test_imputed[cols_to_scale])

print("Scaling complete.")
print("\nSample of scaled training data:")
print(X_train_imputed[cols_to_scale].describe().round(2))

Scaling complete.

Sample of scaled training data:
          age  trestbps    chol  thalch  oldpeak
count  644.00    644.00  644.00  644.00   644.00
mean    -0.00     -0.00    0.00    0.00    -0.00
std      1.00      1.00    1.00    1.00     1.00
min     -2.63     -6.88   -1.82   -2.96    -3.28
25%     -0.75     -0.60   -0.24   -0.69    -0.83
50%      0.08     -0.08    0.23    0.07    -0.26
75%      0.71      0.44    0.62    0.70     0.59
max      2.48      3.58    3.70    2.58     5.02


In [35]:
# Save all preprocessing objects
imputer_path = f'{file_path}/outputs/models/knn_imputer.pkl'
scaler_path = f'{file_path}/outputs/models/standard_scaler.pkl'
bounds_path = f'{file_path}/outputs/models/chol_bounds.pkl'

with open(imputer_path, 'wb') as f:
    pickle.dump(imputer, f)

with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

with open(bounds_path, 'wb') as f:
    pickle.dump(chol_bounds, f)

print("Preprocessing objects saved successfully.")
print(f"Imputer saved to: {imputer_path}")
print(f"Scaler saved to: {scaler_path}")
print(f"Cholesterol bounds saved to: {bounds_path}")

Preprocessing objects saved successfully.
Imputer saved to: C:\Courses\DAMG6105\Project - Disease Prediction System/outputs/models/knn_imputer.pkl
Scaler saved to: C:\Courses\DAMG6105\Project - Disease Prediction System/outputs/models/standard_scaler.pkl
Cholesterol bounds saved to: C:\Courses\DAMG6105\Project - Disease Prediction System/outputs/models/chol_bounds.pkl


In [ ]:
# Save all splits to processed folder
X_train_imputed.to_csv(f'{file_path}/data/processed/X_train.csv', index=False)
X_val_imputed.to_csv(f'{file_path}/data/processed/X_val.csv', index=False)
X_test_imputed.to_csv(f'{file_path}/data/processed/X_test.csv', index=False)

y_train.to_csv(f'{file_path}/data/processed/y_train.csv', index=False)
y_val.to_csv(f'{file_path}/data/processed/y_val.csv', index=False)
y_test.to_csv(f'{file_path}/data/processed/y_test.csv', index=False)

print("All splits saved successfully.")

## Preprocessing Summary

### Columns Removed
- `dataset` - data source identifier, not a clinical feature
- `id` - row identifier, not a clinical feature

### Missing Value Strategy
- `ca` and `thal` were retained despite high missingness (~66% and ~53%) due to their clinical significance
- Missing indicators (`ca_was_missing`, `thal_was_missing`) were added before any transformations to preserve missingness as a signal
- KNN Imputation (k=5) was applied post-split to prevent data leakage
- The imputer was fit exclusively on training data and applied to validation and test sets

### Encoding
- Binary columns (`sex`, `fbs`, `exang`) → mapped to 0/1
- Multi-category columns (`cp`, `restecg`, `slope`) → ordinal encoding
- `thal` → encoded before imputation as KNN requires numeric input

### Final Splits (stratified by target)
- Train: 644 rows (70%)
- Validation: 138 rows (15%)
- Test: 138 rows (15%)

### Feature Scaling
- StandardScaler was applied to continuous numerical features only:  `age`, `trestbps`, `chol`, `thalch`, `oldpeak`
- Binary and categorical columns were excluded from scaling to preserve their meaning.
- Scaler was fit exclusively on training data and applied to validation and test sets to prevent data leakage.
  
### Saved Files
- X_train.csv, X_val.csv, X_test.csv
- y_train.csv, y_val.csv, y_test.csv